# Natural language processing (NLP)

This notebook turns a short example text into numerical features. We explore cleaning, tokenization, stopword removal, lemmatization, stemming, and Bag of Words. Run the cells in order and compare the intermediate outputs.

## 1. Prepare the environment

This notebook uses NLTK for text preprocessing and scikit-learn for Bag of Words. The next cell installs NLTK if it is missing. Make sure scikit-learn is also installed in the notebook environment. NLTK downloads language resources, which requires internet access on the first run.

In [ ]:
!pip install nltk


### Download NLTK language data

This cell downloads the language resources NLTK needs:

- **`stopwords`**: lists of common words, such as *the* and *and*, that you can remove from text.
- **`punkt`**: models for splitting text into sentences.
- **`wordnet`**: a lexical database used for lemmatization, for example turning *cars* into *car*.
- **`punkt_tab`**: additional data for the Punkt tokenizer in newer NLTK versions.

If the resources are already installed and up to date, NLTK does not download them again.


In [ ]:
# Cell 1: Import libraries
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer, PorterStemmer

# Download required NLTK data
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('punkt_tab')


## 2. Inspect the example text

The input consists of four sentences about students and their subjects. This small, hand-written example makes the transformations easy to follow; it is not a dataset for measuring model performance.

In [ ]:
# Cell 2: Example text
text = """
Alice studies Mathematics and Physics. Bob is enrolled in Computer Science and AI.
Carol loves studying English and History. David is taking courses in Biology and Chemistry.
"""
print("Original Text:\n", text)


## 3. Normalize the text

Lowercasing makes words with different capitalization share the same representation. This reduces vocabulary size, but also removes capitalization cues such as proper names.

In [ ]:
# Convert to lowercase
text_clean = text.lower()
print("Cleaned Text:\n", text_clean)

### Remove numbers and punctuation

The regular expression in the next cell removes digit sequences. The example contains no digits, so this step has no visible effect. The print statement displays `text_clean`, while the result is stored in `text_no_numbers`.

The following cell removes characters other than word characters and whitespace. Removing punctuation also removes sentence boundaries, which affects sentence tokenization later.

In [ ]:
# Remove numbers
text_no_numbers = re.sub(r'\d+', '',text_clean )
print("Cleaned Text:\n", text_clean)

In [ ]:
# Cell 3: Text Preprocessing - Lowercasing, removing numbers and punctuation


# Remove punctuation
text_clean = re.sub(r'[^\w\s]', '', text_no_numbers)

print("Cleaned Text:\n", text_clean)


## 4. Split text into tokens

Sentence tokenization divides text into sentences, and word tokenization divides each sentence into tokens. The nested list comprehension creates one list of tokens per detected sentence.

Here, sentence punctuation has already been removed, so the original four sentences are generally treated as one text segment. The second Bag of Words example later starts from the original text to preserve those boundaries.

In [ ]:
# Cell 4: Tokenization
from nltk.tokenize import word_tokenize, sent_tokenize

# Sentence tokenization
sentences = sent_tokenize(text_clean)
print("Tokenized Sentences:\n", sentences)

# Word tokenization
words = [word_tokenize(sentence) for sentence in sentences]
print("\nTokenized Words:\n", words)


### Optional Python syntax examples

The commented examples below show a loop and an equivalent list comprehension for doubling numbers. They illustrate the syntax used in token processing and do not affect the NLP pipeline.

In [ ]:
#numbers = [1, 2, 3, 4, 5]

#doubled = []
#for n in numbers:
    #doubled.append(n * 2)

#print(doubled)


In [ ]:
#numbers = [1, 2, 3, 4, 5]

#doubled = [n * 2 for n in numbers]

#print(doubled)


## 5. Remove stopwords

Stopwords are common words such as ?and? or ?in?. The code filters each token list using the English stopword set. This can emphasize content words, but the choice depends on the task: removing words such as ?not? can change meaning.

In [ ]:
# Cell 5: Remove Stopwords
stop_words = set(stopwords.words('english'))

words_no_stop = []
for sentence in words:
    filtered = [w for w in sentence if w not in stop_words]
    words_no_stop.append(filtered)

print("Words after Stopword Removal:\n", words_no_stop)


## 6. Compare lemmatization and stemming

Lemmatization aims to map a word to its dictionary form. This call does not supply a part of speech, so the lemmatizer uses its default noun interpretation; verbs may remain unchanged. Inspect the output instead of assuming every inflected word becomes its base form.

In [ ]:
# Cell 6: Lemmatization
lemmatizer = WordNetLemmatizer()

words_lemmatized = []
for sentence in words_no_stop:
    lem_sentence = [lemmatizer.lemmatize(w) for w in sentence]
    words_lemmatized.append(lem_sentence)

print("Lemmatized Words:\n", words_lemmatized)


### Stemming

Porter stemming applies rules to shorten words, and the result need not be a dictionary word. Both this step and lemmatization start from `words_no_stop`, so they are alternative transformations. The next Bag of Words example uses the stemmed version.

In [ ]:
# Cell 7: Stemming
stemmer = PorterStemmer()

words_stemmed = []
for sentence in words_no_stop:
    stem_sentence = [stemmer.stem(w) for w in sentence]
    words_stemmed.append(stem_sentence)

print("Stemmed Words:\n", words_stemmed)


## 7. Build a Bag of Words matrix

The token lists are joined into strings before vectorization. `CountVectorizer` learns a vocabulary and counts its terms: rows represent text segments, columns represent vocabulary terms, and values represent occurrence counts. Match the displayed vocabulary to the matrix columns.

Bag of Words does not preserve word order or directly represent meaning. Because sentence boundaries were removed earlier, this first matrix generally has one row.

In [ ]:
# Cell 8: Bag of Words from Stemmed Words
from sklearn.feature_extraction.text import CountVectorizer

# Join stemmed words back into sentences
sentences_stemmed = [" ".join(sentence) for sentence in words_stemmed]

# Initialize CountVectorizer
vectorizer = CountVectorizer()
X_bow = vectorizer.fit_transform(sentences_stemmed)

# Convert BoW matrix to array and display
print("Vocabulary:", vectorizer.get_feature_names_out())
print("BoW Matrix:\n", X_bow.toarray())


## 8. Preserve sentences before vectorization

This second example restores the original text and detects sentences before removing punctuation. Compare the four detected sentences with the earlier tokenization output.

In [ ]:
# Sample text
text = """Alice studies Mathematics and Physics. Bob is enrolled in Computer Science and AI.
Carol loves studying English and History. David is taking courses in Biology and Chemistry."""

# Step 1: Sentence Tokenization
sentences = sent_tokenize(text)
print("Sentences:\n", sentences)


### Compare the sentence-level matrix

Each original sentence now becomes a separate row. This example does not apply the earlier stopword removal or stemming. The vectorizer applies its own default text processing, so the vocabulary differs from the first example.

Check which terms occur in several rows and which identify a particular sentence. The matrix could serve as input to a later model, but this notebook does not train or evaluate a classifier.

In [ ]:
# Step 2: Word Tokenization and join tokens back into sentences
words_tokenized = [word_tokenize(sentence) for sentence in sentences]
sentences_for_bow = [" ".join(tokens) for tokens in words_tokenized]
# Step 3: Create Bag of Words model
vectorizer = CountVectorizer()
X_bow = vectorizer.fit_transform(sentences_for_bow)

# Step 4: Display vocabulary and BoW matrix
print("Vocabulary:", vectorizer.get_feature_names_out())
print("\nBoW Matrix:\n", X_bow.toarray())